# CancelVoice: Privacy-Preserving Voice Anonymization Demo

This notebook demonstrates the core anonymization pipeline from **CancelVoice**, a privacy-preserving voice authentication system using adversarially-trained generative models and diffusion-based privacy filters.

We explore two fundamental blurring strategies adapted for speaker identity suppression:
- **Low-pass blurring** — removes high-frequency phonetic features that carry speaker identity cues (§2.2.1)
- **MFCC inversion** — reconstructs speech from a reduced MFCC representation, stripping fine-grained spectral characteristics (§2.2.2)
- **Pipeline (combined)** — applies both stages sequentially as a baseline privacy filter before the diffusion model stage

These blurring methods serve as the **pre-processing baseline** in CancelVoice, prior to the adversarially-trained anonymization layer.

---
**Reference:** Cohen-Hadria et al. (2019). *Voice Anonymization.* [PDF](https://markcartwright.com/files/cohen-hadria2019voiceanonymization.pdf)

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import librosa
import numpy as np
from IPython.display import Audio, display

# Resolve repo root — works whether kernel is launched from repo root or notebooks/
for _root in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (_root / "voice_anonymization").is_dir():
        sys.path.insert(0, str(_root))
        break

from voice_anonymization import low_pass_blur, mfcc_inversion_blur
# ------------------------------------------------------------------
# Audio input — set AUDIO_PATH to a speaker WAV file for real evaluation.
# Recommended: use a VoxCeleb1 clip from your evaluation split.
# Falls back to a synthetic 200 Hz tone if no file is provided.
# ------------------------------------------------------------------
_audio = os.environ.get("AUDIO_PATH", "").strip()
_audio = _audio or "demo.wav"  # place your speaker clip here
AUDIO_PATH = Path(_audio)

if AUDIO_PATH.is_file():
    y, sr = librosa.load(AUDIO_PATH, mono=True, sr=None)
    print(f"Loaded speaker clip: {AUDIO_PATH.resolve()}")
    print(f"Sample rate: {sr} Hz | Duration: {len(y) / sr:.2f}s | Samples: {len(y)}")
else:
    sr = 16000
    t = np.linspace(0.0, 2.0, int(2 * sr), endpoint=False, dtype=np.float32)
    y = (0.2 * np.sin(2.0 * np.pi * 200.0 * t)).astype(np.float32)
    print("[INFO] No AUDIO_PATH found — using synthetic 200 Hz tone as stand-in.")
    print("       Set AUDIO_PATH=/path/to/voxceleb_clip.wav for speaker identity evaluation.")

print("\nOriginal (pre-anonymization):")
display(Audio(y, rate=sr))

[INFO] No AUDIO_PATH found — using synthetic 200 Hz tone as stand-in.
       Set AUDIO_PATH=/path/to/voxceleb_clip.wav for speaker identity evaluation.

Original (pre-anonymization):


## Stage 1 — Low-Pass Blurring

Downsamples to 500 Hz then resamples back to the original rate. This removes high-frequency formant and prosodic features that correlate with speaker identity, while preserving coarse phoneme intelligibility.

In the CancelVoice pipeline, this acts as a deterministic baseline filter — the adversarial generator then refines the output to maintain authentication utility.

In [2]:
# Stage 1: Low-pass blurring — strips high-frequency speaker identity cues
y_lp, sr_lp = low_pass_blur(y, sr)

print(f"Low-pass blurred output: sr={sr_lp} Hz | samples={len(y_lp)}")
print("Spectral content above ~500 Hz suppressed — speaker timbre significantly altered.")
display(Audio(y_lp, rate=sr_lp))

Low-pass blurred output: sr=16000 Hz | samples=32000
Spectral content above ~500 Hz suppressed — speaker timbre significantly altered.


## Stage 2 — MFCC Inversion Blurring

Extracts the first 5 MFCCs (capturing coarse spectral envelope), converts to a mel spectrogram, then reconstructs via Griffin-Lim. This discards fine-grained spectral texture — a key carrier of speaker identity.

The reconstruction typically differs in length from the input due to STFT windowing — this is expected behaviour and handled downstream in the full CancelVoice pipeline.

In [3]:
# Stage 2: MFCC inversion — coarse spectral envelope only (5 MFCCs → Griffin-Lim reconstruction)
y_mfcc = mfcc_inversion_blur(y, sr)

print(f"MFCC inversion output: samples={len(y_mfcc)} (length drift from Griffin-Lim is expected)")
print("Speaker-specific fine spectral texture suppressed — phoneme content partially preserved.")
display(Audio(y_mfcc, rate=sr))

MFCC inversion output: samples=31744 (length drift from Griffin-Lim is expected)
Speaker-specific fine spectral texture suppressed — phoneme content partially preserved.


## Stage 3 — Full Blurring Pipeline (Low-Pass → MFCC Inversion)

Applies both stages sequentially. This represents the **baseline anonymization floor** for CancelVoice — the minimum speaker identity suppression before the diffusion-based privacy filter is applied.

In ablation studies, this pipeline output is compared against the adversarially-trained model to quantify the privacy-utility tradeoff introduced by each component.

In [4]:
# Stage 3: Combined pipeline — low-pass first, then MFCC inversion
y_lp2, sr_lp2 = low_pass_blur(y, sr)
y_chain = mfcc_inversion_blur(y_lp2, float(sr_lp2))

print(f"Pipeline output: sr={sr_lp2} Hz | final samples={len(y_chain)}")
print("Both speaker cue channels suppressed — strongest baseline anonymization.")
display(Audio(y_chain, rate=sr_lp2))

Pipeline output: sr=16000 Hz | final samples=31744
Both speaker cue channels suppressed — strongest baseline anonymization.


## Spectral Analysis — Comparing Anonymization Stages

Visualize the log-mel spectrogram before and after each anonymization stage. Speaker identity information manifests primarily as fine spectral texture in the upper frequency bands — inspect how each stage degrades this.

In [5]:
import matplotlib.pyplot as plt

def plot_mel(signal, sample_rate, title, ax):
    """Plot log-mel spectrogram for visual privacy inspection."""
    S = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, x_axis="time", y_axis="mel", sr=sample_rate,
                                   fmax=8000, ax=ax)
    ax.set_title(title, fontsize=10)
    return img

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("CancelVoice — Mel Spectrogram Comparison Across Anonymization Stages",
             fontsize=12, fontweight="bold")

plot_mel(y, sr, "Original (No Anonymization)", axes[0, 0])
plot_mel(y_lp, sr_lp, "Stage 1: Low-Pass Blurring (500 Hz cutoff)", axes[0, 1])
plot_mel(y_mfcc, sr, "Stage 2: MFCC Inversion (5 MFCCs)", axes[1, 0])
plot_mel(y_chain, sr_lp2, "Stage 3: Pipeline (Low-Pass → MFCC)", axes[1, 1])

plt.tight_layout()
plt.savefig("cancelvoice_spectrogram_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Spectrogram saved to cancelvoice_spectrogram_comparison.png")

Spectrogram saved to cancelvoice_spectrogram_comparison.png


## Privacy Metric: Speaker Embedding Distance

Compute cosine distance between the original and anonymized speaker embeddings using `speechbrain` x-vectors. Larger distance → greater speaker identity suppression.

> **Note:** Install speechbrain if not present: `pip install speechbrain`

This metric is used in CancelVoice evaluation to compare the blurring baseline against the full adversarial anonymization model.

In [6]:
try:
    import torch
    from speechbrain.pretrained import EncoderClassifier

    classifier = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-xvect-voxceleb",
        savedir="/tmp/xvect_model"
    )

    def get_embedding(signal, sample_rate):
        tensor = torch.tensor(signal).unsqueeze(0)
        with torch.no_grad():
            emb = classifier.encode_batch(tensor)
        return emb.squeeze().numpy()

    def cosine_distance(a, b):
        return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    emb_orig  = get_embedding(y, sr)
    emb_lp    = get_embedding(y_lp, sr_lp)
    emb_mfcc  = get_embedding(y_mfcc, sr)
    emb_chain = get_embedding(y_chain, sr_lp2)

    print("Speaker embedding cosine distances from original (↑ = more anonymized)")
    print(f"  Low-pass blurring:       {cosine_distance(emb_orig, emb_lp):.4f}")
    print(f"  MFCC inversion:          {cosine_distance(emb_orig, emb_mfcc):.4f}")
    print(f"  Pipeline (both stages):  {cosine_distance(emb_orig, emb_chain):.4f}")

except ImportError:
    print("[INFO] speechbrain not installed — skipping embedding distance evaluation.")
    print("       Install with: pip install speechbrain")
    print("       This metric is used in CancelVoice ablation studies to quantify privacy gains.")

[INFO] speechbrain not installed — skipping embedding distance evaluation.
       Install with: pip install speechbrain
       This metric is used in CancelVoice ablation studies to quantify privacy gains.
